# Funding Calls Ingestion

This notebook is only for bringing PDF data into the system. It reads local files, extracts text, creates chunks, builds embeddings, and writes them into a persistent ChromaDB collection.

## Setup

Install required packages in the active environment.

In [1]:
%pip install -q chromadb pypdf sentence-transformers tqdm

Note: you may need to restart the kernel to use updated packages.


## Imports

In [2]:
from pathlib import Path
import hashlib
import re
from typing import Dict, List

from pypdf import PdfReader
from tqdm.auto import tqdm

import chromadb
from chromadb.utils import embedding_functions

/home/lburmazevic/Projects/EY-AI-Techniques-Solution/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Paths and chunking settings are centralized here for easy reruns.

In [3]:
PROJECT_ROOT = Path.cwd()
PDF_DIR = PROJECT_ROOT / "docs" / "fundingcalls"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "funding_calls"

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 250
MIN_CHUNK_CHARS = 120

RESET_COLLECTION = True
EXCLUDE_FILES = set()

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF directory: {PDF_DIR}")
print(f"Chroma directory: {CHROMA_DIR}")

Project root: /home/lburmazevic/Projects/EY-AI-Techniques-Solution
PDF directory: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/fundingcalls
Chroma directory: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/data/chroma


## Source Files

List the PDF files that will be ingested.

In [4]:
pdf_paths = sorted([p for p in PDF_DIR.glob("*.pdf") if p.name not in EXCLUDE_FILES])

print(f"Found {len(pdf_paths)} PDF files")
for path in pdf_paths:
    print(" -", path.name)

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {PDF_DIR}")

Found 6 PDF files
 - Centri Nazionali Tematiche PNRR.pdf
 - Ecosistemi dell'Innovazione PNRR.pdf
 - HORIZON_WIDERA_ERA_calls_merged.pdf
 - PRIN 2022 PNRR.pdf
 - PRIN PNRR 2022 NextGen.pdf
 - Partenariati Estesi PNRR.pdf


## Text Extraction

Text is extracted page by page and lightly cleaned before chunking.

In [5]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_pages_from_pdf(pdf_path: Path) -> List[Dict]:
    rows: List[Dict] = []
    reader = PdfReader(str(pdf_path))

    for page_idx, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        text = clean_text(raw_text)

        if len(text) < 40:
            continue

        rows.append(
            {
                "source_file": pdf_path.name,
                "source_path": str(pdf_path),
                "page": page_idx,
                "text": text,
            }
        )

    return rows

In [6]:
page_rows: List[Dict] = []
for pdf_path in tqdm(pdf_paths, desc="Extracting PDF pages"):
    page_rows.extend(extract_pages_from_pdf(pdf_path))

print(f"Extracted {len(page_rows)} non-empty pages")
page_rows[0] if page_rows else "No text extracted"

Extracting PDF pages: 100%|██████████| 6/6 [00:08<00:00,  1.48s/it]

Extracted 329 non-empty pages


{'source_file': 'Centri Nazionali Tematiche PNRR.pdf',
 'source_path': '/home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/fundingcalls/Centri Nazionali Tematiche PNRR.pdf',
 'page': 1,
 'text': '1 Avviso pubblico per la presentazione di Proposte di intervento per il Potenziamento di strutture di ricerca e creazione di “campioni nazionali” di R&S su alcune Key Enabling Technologies da finanziare nell’ambito del Piano Nazionale di Ripresa e Resilienza, Missione 4 Componente 2 Investimento 1.4 finanziato dall’Unione europea – NextGenerationEU. Allegato A - Tematiche (articolo 1 comma 1 dell’Avviso)'}

## Chunking

Overlapping chunks are created to improve retrieval around text boundaries.

In [7]:
def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if overlap >= size:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")

    chunks: List[str] = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + size, text_len)
        chunk = text[start:end].strip()

        if len(chunk) >= MIN_CHUNK_CHARS:
            chunks.append(chunk)

        if end == text_len:
            break

        start = end - overlap

    return chunks

In [8]:
documents: List[str] = []
metadatas: List[Dict] = []
ids: List[str] = []

for row in tqdm(page_rows, desc="Chunking pages"):
    chunks = chunk_text(row["text"])

    for chunk_idx, chunk in enumerate(chunks):
        raw_id = f"{row['source_file']}|{row['page']}|{chunk_idx}|{len(chunk)}"
        chunk_id = hashlib.md5(raw_id.encode("utf-8")).hexdigest()

        documents.append(chunk)
        metadatas.append(
            {
                "source_file": row["source_file"],
                "source_path": row["source_path"],
                "page": row["page"],
                "chunk_index": chunk_idx,
                "chunk_chars": len(chunk),
            }
        )
        ids.append(chunk_id)

print(f"Created {len(documents)} chunks")
if documents:
    print("Sample chunk preview:")
    print(documents[0][:500])

Chunking pages: 100%|██████████| 329/329 [00:00<00:00, 101204.69it/s]

Created 682 chunks
Sample chunk preview:
1 Avviso pubblico per la presentazione di Proposte di intervento per il Potenziamento di strutture di ricerca e creazione di “campioni nazionali” di R&S su alcune Key Enabling Technologies da finanziare nell’ambito del Piano Nazionale di Ripresa e Resilienza, Missione 4 Componente 2 Investimento 1.4 finanziato dall’Unione europea – NextGenerationEU. Allegato A - Tematiche (articolo 1 comma 1 dell’Avviso)


## ChromaDB Ingestion

Chunks are embedded and written to a persistent collection.

In [9]:
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-mpnet-base-v2"
)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if RESET_COLLECTION:
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn,
    metadata={"description": "Funding call chunks for RAG retrieval"},
)

BATCH_SIZE = 128
for i in tqdm(range(0, len(documents), BATCH_SIZE), desc="Upserting to Chroma"):
    collection.upsert(
        ids=ids[i:i + BATCH_SIZE],
        documents=documents[i:i + BATCH_SIZE],
        metadatas=metadatas[i:i + BATCH_SIZE],
    )

print("Collection name:", COLLECTION_NAME)
print("Stored chunks:", collection.count())

Upserting to Chroma: 100%|██████████| 6/6 [01:08<00:00, 11.35s/it]

Collection name: funding_calls
Stored chunks: 682


## Ingestion Validation

This check confirms that the number of generated chunks matches what was written to ChromaDB.

In [10]:
expected_chunks = len(documents)
stored_chunks = collection.count()

print("Expected chunks:", expected_chunks)
print("Stored chunks:", stored_chunks)
print("Unique IDs:", len(set(ids)))
print("Count match:", expected_chunks == stored_chunks)

if expected_chunks != stored_chunks:
    raise ValueError("Chunk count mismatch: some chunks may not have been written to ChromaDB.")

Expected chunks: 682
Stored chunks: 682
Unique IDs: 682
Count match: True


## Optional Retrieval Check

A quick query is useful to verify that semantic search is working and metadata is attached correctly.

In [ ]:
query = "startup deep tech and innovation funding"
results = collection.query(query_texts=[query], n_results=3)

for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1,
):
    print(f"\nResult {i} | distance={dist:.4f}")
    print(meta)
    print(doc[:450], "...")


Result 1 | distance=0.3451
{'chunk_index': 1, 'chunk_chars': 1250, 'page': 20, 'source_path': '/home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/fundingcalls/Partenariati Estesi PNRR.pdf', 'source_file': 'Partenariati Estesi PNRR.pdf'}
a meno di 5 anni, Startup innovative e Spin off da ricerca (anche in termini di cofinanziamento) di tipo scientifico, tecnologico, culturale e della società civile; coinvolgimento di attori di diverse regioni e diverse zone del paese, compreso il Mezzogiorno e le isole 5 10 - qualità dei dati e degli indicatori (milestones e target intermedi e finali) proposti per il monitoraggio delle attività . 5 10 C) Impatto del programma 15 30 - Analisi del  ...

Result 2 | distance=0.3925
{'source_file': 'Partenariati Estesi PNRR.pdf', 'source_path': '/home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/fundingcalls/Partenariati Estesi PNRR.pdf', 'chunk_chars': 1394, 'page': 12, 'chunk_index': 1}
rso nel rispetto delle norme comunitarie e nazionali ap